This notebook provides examples to go along with the [textbook](http://manipulation.csail.mit.edu/robot.html).  I recommend having both windows open, side-by-side!

In [1]:
from pydrake.all import ModelVisualizer, PackageMap, Simulator, StartMeshcat

from manipulation import ConfigureParser, FindResource, running_as_notebook
from manipulation.remotes import AddSpotRemote
from manipulation.station import LoadScenario, MakeHardwareStation

In [2]:
# Start the visualizer.
meshcat = StartMeshcat()

INFO:drake:Meshcat listening for connections at http://localhost:7001


# Simplified Spot model for mobile manipulation

First we'll use the ModelVisualizer to inspect the model.

In [3]:
visualizer = ModelVisualizer(meshcat=meshcat)
ConfigureParser(visualizer.parser())
AddSpotRemote(visualizer.parser().package_map())
visualizer.AddModels(
    url="package://manipulation/spot/spot_with_arm_and_floating_base_actuators.urdf"
)
visualizer.Run(loop_once=not running_as_notebook)
meshcat.DeleteAddedControls()

INFO:drake:PackageMap: Downloading https://github.com/wrangel-bdai/spot_ros2/archive/20965ef7bba98598ee10878c7b54e6ef28a300c6.tar.gz
==== LCM Warning ===
LCM detected that large packets are being received, but the kernel UDP
receive buffer is very small.  The possibility of dropping packets due to
insufficient buffer space is very high.

For more information, visit:
   https://lcm-proj.github.io/lcm/content/multicast-setup.html

INFO:drake:Click 'Stop Running' or press Esc to quit


Now we can use HardwareStation to create a basic simulation.

In [4]:
scenario = LoadScenario(
    filename=FindResource(
        "models/spot/spot_with_arm_and_floating_base_actuators.scenario.yaml"
    )
)
station = MakeHardwareStation(
    scenario,
    meshcat,
    parser_preload_callback=lambda parser: AddSpotRemote(parser.package_map()),
)
simulator = Simulator(station)
context = simulator.get_mutable_context()
x0 = station.GetOutputPort("spot.state_estimated").Eval(context)
station.GetInputPort("spot.desired_state").FixValue(context, x0)
simulator.AdvanceTo(0.1);

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=e969edb2-a7e6-480b-b81d-4d5778c62ec9' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>